# Rally E4B Direct Text Export

Exports the direct Heretic E4B text browser package without RP merge.

In [ ]:
import os, subprocess, sys, time

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
secret_token = ''
for attempt in range(5):
    try:
        from kaggle_secrets import UserSecretsClient
        secret_token = UserSecretsClient().get_secret('HF_TOKEN')
        break
    except Exception:
        time.sleep(3)
if secret_token:
    os.environ.setdefault('HF_TOKEN', secret_token)
packages = [
    'git+https://github.com/huggingface/transformers.git',
    'accelerate>=1.13.0', 'huggingface_hub[cli]>=1.5.0', 'hf_transfer>=0.1.9',
    'safetensors>=0.7.0', 'onnx>=1.19.0', 'onnxruntime>=1.23.0', 'onnxscript>=0.5.0',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', *packages])


In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

prep_root = next(
    path for path in [
        Path('/kaggle/input/rally-e4b-export-prep/rally-e4b-export-prep'),
        Path('/kaggle/input/rally-e4b-export-prep'),
        Path('/kaggle/input/notebooks/thomasjvu/rally-e4b-export-prep/rally-e4b-export-prep'),
    ]
    if path.exists()
)
staged_repo = prep_root / 'heretic-to-onnx'
staged_template = prep_root / 'optimized-template'
REPO_DIR = Path('/kaggle/working/heretic-to-onnx')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
shutil.copytree(staged_repo, REPO_DIR)

WORK_DIR = Path('/kaggle/working/rally-e4b-direct-text-export')
REPORT_PATH = WORK_DIR / 'rally-e4b-direct-export-report.json'
cmd = [
    sys.executable, str(REPO_DIR / 'scripts/kaggle_rally_e2b_two_stage_export.py'),
    '--work-dir', str(WORK_DIR),
    '--report-path', str(REPORT_PATH),
    '--scratch-dir', '/kaggle/temp/rally-e4b-direct-text-export',
    '--direct-source-model-id', os.environ.get('RALLY_HERETIC_MODEL_ID', 'coder3101/gemma-4-E4B-it-heretic'),
    '--base-model-id', os.environ.get('RALLY_BASE_MODEL_ID', 'google/gemma-4-E4B-it'),
    '--direct-text-repo', os.environ.get('RALLY_DIRECT_TEXT_REPO', 'thomasjvu/rally-4b-text'),
    '--export-device', os.environ.get('RALLY_EXPORT_DEVICE', 'cpu'),
    '--optimized-template-dir', str(staged_template),
    '--optimized-template-model-id', 'onnx-community/gemma-4-E4B-it-ONNX',
    '--direct-full-template', 'configs/heretic-to-onnx.gemma4-e4b-heretic.yaml',
    '--direct-text-template', 'configs/heretic-to-onnx.gemma4-e4b-heretic-text.yaml',
    '--skip-rp', '--skip-full-packages', '--no-score',
]
if os.environ.get('RALLY_UPLOAD', '0') != '1':
    cmd.append('--no-upload')
subprocess.check_call(cmd)
